# Distancia a la Moneda Ideal y Posicionamiento Global

**Tesis de licenciatura** — Marco Antonio López Lázaro

**Instrucciones:** Click en `Runtime → Run all` (o Ctrl+F9).

Tarda 3-5 minutos. Al final muestra todos los resultados.

In [ ]:
# ============================================================
# PASO 1: Clonar el repositorio y entrar a la carpeta
# ============================================================
import os

if not os.path.exists('moneda-ideal-posicionamiento'):
    !git clone https://github.com/marcovanfan101/moneda-ideal-posicionamiento.git

%cd /content/moneda-ideal-posicionamiento
print('Carpeta actual:', os.getcwd())
!ls -la

In [ ]:
# ============================================================
# PASO 2: Instalar dependencias
# ============================================================
!pip install -q yfinance pandas numpy pandas-datareader requests scipy openpyxl matplotlib seaborn
print('Dependencias instaladas')

In [ ]:
# ============================================================
# PASO 3: Ejecutar el pipeline completo
# (Esto descarga los datos y calcula todos los indices)
# ============================================================
!python scripts/01_run_all.py

In [ ]:
# ============================================================
# PASO 4: Mostrar la tabla final
# ============================================================
import pandas as pd
import numpy as np

df = pd.read_excel('data/processed/datos_completos.xlsx', index_col=0)

print('=' * 80)
print('TABLA FINAL: DISTANCIA A LA MONEDA IDEAL')
print('=' * 80)
print()

cols = ['Eo', 'Sv', 'SM2', 'D', 'Reservas_COFER', 'BIS_Turnover']
df_sorted = df.dropna(subset=['D']).sort_values('D')
print(df_sorted[cols].round(4).to_string())

In [ ]:
# ============================================================
# PASO 5: Analisis del umbral D = 0.10
# ============================================================
print('=' * 80)
print('ANALISIS DEL UMBRAL D = 0.10')
print('=' * 80)
print()

cerca = df_sorted[df_sorted['D'] < 0.10]
lejos = df_sorted[df_sorted['D'] >= 0.10]

print(f'Divisas con D < 0.10 ({len(cerca)}):')
print(f'  Reservas COFER totales: {cerca["Reservas_COFER"].sum():.2f}%')
print(f'  BIS Turnover total: {cerca["BIS_Turnover"].sum():.2f}%')
print()
print(f'Divisas con D >= 0.10 ({len(lejos)}):')
print(f'  Reservas COFER totales: {lejos["Reservas_COFER"].sum():.2f}%')
print(f'  BIS Turnover total: {lejos["BIS_Turnover"].sum():.2f}%')

In [ ]:
# ============================================================
# PASO 6: Correlaciones
# ============================================================
corr = pd.read_excel('results/tables/correlaciones.xlsx')
print('CORRELACIONES: D vs POSICIONAMIENTO GLOBAL')
print('=' * 80)
print(corr.round(4).to_string(index=False))

In [ ]:
# ============================================================
# PASO 7: Regresiones
# ============================================================
reg = pd.read_excel('results/tables/regresiones.xlsx')
print('REGRESIONES: log(posicionamiento) ~ D')
print('=' * 80)
print(reg.round(4).to_string(index=False))

In [ ]:
# ============================================================
# PASO 8: Robustez (sin Turquia, sin Argentina)
# ============================================================
robust = pd.read_excel('results/tables/robustez.xlsx')
print('ANALISIS DE ROBUSTEZ')
print('=' * 80)
print(robust.round(4).to_string(index=False))

In [ ]:
# ============================================================
# PASO 9: Grafico
# ============================================================
import matplotlib.pyplot as plt

df_plot = df.dropna(subset=['D', 'Reservas_COFER']).copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
data = df_plot[df_plot['Reservas_COFER'] > 0]
ax.scatter(data['D'], data['Reservas_COFER'], c='steelblue', s=100, alpha=0.7)
for idx, row in data.iterrows():
    ax.annotate(idx, (row['D'], row['Reservas_COFER']), fontsize=9)
ax.set_yscale('log')
ax.set_xlabel('D = (100-Eo) * Sv * SM2')
ax.set_ylabel('Reservas COFER (%, log)')
ax.set_title('Distancia al Ideal vs Reservas')
ax.grid(True, alpha=0.3)
ax.axvline(x=0.10, color='red', linestyle='--', alpha=0.5, label='D=0.10')
ax.legend()

ax = axes[1]
data = df_plot[df_plot['BIS_Turnover'] > 0]
ax.scatter(data['D'], data['BIS_Turnover'], c='darkred', s=100, alpha=0.7)
for idx, row in data.iterrows():
    ax.annotate(idx, (row['D'], row['BIS_Turnover']), fontsize=9)
ax.set_yscale('log')
ax.set_xlabel('D = (100-Eo) * Sv * SM2')
ax.set_ylabel('BIS Turnover (%, log)')
ax.set_title('Distancia al Ideal vs Turnover BIS')
ax.grid(True, alpha=0.3)
ax.axvline(x=0.10, color='red', linestyle='--', alpha=0.5, label='D=0.10')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 10: Resumen ejecutivo
# ============================================================
print('=' * 80)
print('RESUMEN EJECUTIVO')
print('=' * 80)
print()
print(f'Divisas con los 3 indices: {len(df_sorted)}')
print(f'Periodo: 2010-2025')
print()
print('HALLAZGO PRINCIPAL:')
print(f'  Umbral en D = 0.10')
print(f'  Divisas con D < 0.10: {len(cerca)}')
print(f'  Reservas concentradas: {cerca["Reservas_COFER"].sum():.2f}%')
print(f'  Divisas con D >= 0.10: {len(lejos)}')
print(f'  Reservas del resto: {lejos["Reservas_COFER"].sum():.2f}%')
print()
print('CORRELACIONES:')
print(f'  D vs Reservas: rho = {corr.iloc[0]["Spearman_rho"]:.4f} (p = {corr.iloc[0]["Spearman_p"]:.4f})')
print(f'  D vs BIS:      rho = {corr.iloc[1]["Spearman_rho"]:.4f} (p = {corr.iloc[1]["Spearman_p"]:.4f})')
print()
print('CONCLUSION:')
print('  Existe un umbral empirico en D = 0.10 que separa las divisas')
print('  con posicionamiento global de las que no lo tienen.')